# Batch Unconstraining Results Analysis (Product Level)

This notebook evaluates the performance of different unconstraining models (Naive, EM, MARSS, EM-X Price, MARSS-X Price) across multiple cargo segments and calculates global aggregate performance.

In [1]:
import pandas as pd
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython.display import display

FILTER_ZEROS = False
THRESHOLD = 1
SEGMENTS = ["Contract", "General", "Perishable", "Express", "Spot"]
MODELS = ["Naive", "EM", "MARSS", "EMXPrice", "MARSSXPrice"]

In [2]:
files = glob.glob("./unconstrained_results/*.parquet")

df_all = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

print(f"Loaded {len(files)} files. Shape: {df_all.shape}")
df_all.head()

In [3]:
def trim_burn_in(df_group):
    # Check for NaNs in any of the estimation columns across segments
    # All models share the same LOOKBACK=365
    est_cols = [f"{m}_{s}_Est" for m in MODELS for s in SEGMENTS]
    valid_mask = df_group[est_cols].notna().all(axis=1)
    
    if not valid_mask.any():
        return pd.DataFrame()
        
    first_valid_idx = valid_mask.idxmax() 
    return df_group.loc[first_valid_idx:].copy()

print("Trimming burn-in period...")
df_trimmed = df_all.groupby(['Origin', 'Destination'], as_index=False, group_keys=False).apply(trim_burn_in)

print("Calculating Total metrics across all segments...")
for m in MODELS:
    total_est_col = f"{m}_Total_Est"
    df_trimmed[total_est_col] = 0.0
    
df_trimmed["Total_Oracle_kg"] = 0.0
df_trimmed["Total_Observed_kg"] = 0.0

for s in SEGMENTS:
    obs_col = f"Observed_{s}_kg"
    oracle_col = f"Oracle_{s}_kg"
    
    df_trimmed["Total_Oracle_kg"] += df_trimmed[oracle_col].fillna(0)
    df_trimmed["Total_Observed_kg"] += df_trimmed[obs_col].fillna(0)
    
    for m in MODELS:
        est_col = f"{m}_{s}_Est"
        # Fill remaining NaNs with observed (safety fallback)
        df_trimmed[est_col] = df_trimmed[est_col].fillna(df_trimmed[obs_col])
        df_trimmed[f"{m}_Total_Est"] += df_trimmed[est_col]

print(f"Trimmed shape: {df_trimmed.shape}")
df_analysis = df_trimmed.reset_index(drop=True)

In [4]:
all_results = []
EVAL_SEGMENTS = SEGMENTS + ["Total"]

for seg in EVAL_SEGMENTS:
    true_col = f"Oracle_{seg}_kg" if seg != "Total" else "Total_Oracle_kg"
    
    for model in MODELS:
        est_col = f"{model}_{seg}_Est" if seg != "Total" else f"{model}_Total_Est"
        
        route_metrics = []
        for (o, d), g in df_analysis.groupby(["Origin", "Destination"]):
            yt = g[true_col].values
            yp = g[est_col].values
            
            mask = ~np.isnan(yt) & ~np.isnan(yp)
            if mask.sum() == 0: continue
            
            yt_v = yt[mask]
            yp_v = yp[mask]
            
            mae = np.mean(np.abs(yt_v - yp_v))
            rmse = np.sqrt(np.mean((yt_v - yp_v)**2))
            route_metrics.append((mae, rmse))
            
        if route_metrics:
            arr = np.array(route_metrics)
            all_results.append({
                "Segment": seg,
                "Model": model,
                "Avg_MAE": arr[:, 0].mean(),
                "Avg_RMSE": arr[:, 1].mean()
            })

summary_df = pd.DataFrame(all_results)

print("--- Mean Absolute Error (MAE) by Segment ---")
display(summary_df.pivot(index="Model", columns="Segment", values="Avg_MAE"))

print("\n--- Global Aggregate Metrics (Total across all Products) ---")
display(summary_df[summary_df["Segment"] == "Total"].set_index("Model").drop(columns="Segment"))

In [5]:
lift_results = []
total_obs_vol = df_analysis["Total_Observed_kg"].sum()
total_oracle_vol = df_analysis["Total_Oracle_kg"].sum()

for model in MODELS:
    est_col = f"{model}_Total_Est"
    total_est_vol = df_analysis[est_col].sum()
    
    lift_results.append({
        "Model": model,
        "Total_Volume_Est": total_est_vol,
        "Lift_Factor (Est/Obs)": total_est_vol / total_obs_vol,
        "Accuracy_Ratio (Est/Oracle)": total_est_vol / total_oracle_vol
    })

lift_df = pd.DataFrame(lift_results).set_index("Model")
print("--- Volume & Bias (Overestimation) Analysis ---")
display(lift_df)

In [6]:
fig, ax = plt.subplots(1, 2, figsize=(20, 7))

# Left: Segment MAE
sns.barplot(data=summary_df[summary_df["Segment"] != "Total"], x="Segment", y="Avg_MAE", hue="Model", ax=ax[0])
ax[0].set_title("MAE by Product Segment")
ax[0].grid(axis='y', alpha=0.3)

# Right: Global MAE
sns.barplot(data=summary_df[summary_df["Segment"] == "Total"], x="Model", y="Avg_MAE", ax=ax[1])
ax[1].set_title("Global MAE (Total Demand)")
ax[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()